In [148]:
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from dotenv import load_dotenv
import os
from openai import OpenAI





In [149]:
# Load .env file into environment
load_dotenv()

# NVIDIA
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
NVIDIA_LLM_API_KEY = os.getenv("NVIDIA_LLM_API_KEY")


# Qdrant
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

# Basic safety checks (optional but recommended)
if not NVIDIA_API_KEY:
    raise ValueError("NVIDIA_API_KEY not found in environment")

if not QDRANT_URL or not QDRANT_API_KEY:
    raise ValueError("QDRANT credentials not found in environment")

In [150]:

PDF_PATHS = [
    "pdf_documents/Residence_certificate_info.pdf",
    "pdf_documents/Residence_Certificate_Service_Explanation.pdf"
]

documents = []
for path in PDF_PATHS:
    loader = PyMuPDFLoader(path)
    documents.extend(loader.load())


In [151]:
SECTION_HEADERS = [
    "Service Details",
    "Mandatory Documents",
    "Category-wise Document Classification",
    "General Citizens",
    "Government Employees",
    "Important Notes",
    "What is the Residence Certificate Service",
    "Why Documents are Collected",
    "Reasoning Behind Each Document"
]


In [152]:
def split_by_section(documents):
    sectioned_docs = []

    pattern = r"(?=(" + "|".join(SECTION_HEADERS) + r"))"

    for doc in documents:
        text = doc.page_content
        splits = re.split(pattern, text)

        for chunk in splits:
            cleaned = chunk.strip()
            if cleaned:
                sectioned_docs.append(
                    Document(
                        page_content=cleaned,
                        metadata=doc.metadata
                    )
                )

    return sectioned_docs


In [153]:
adaptive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,          # high precision
    chunk_overlap=60,        # preserves meaning
    separators=[
        "\n\n",              # paragraphs
        "\n",                # lines
        "•", "-", "1.",      # lists
        " "
    ]
)


In [154]:
def efficient_chunking(documents):
    # Step 1: Split by logical sections
    section_docs = split_by_section(documents)

    # Step 2: Chunk within sections
    chunks = adaptive_splitter.split_documents(section_docs)

    return chunks


In [155]:
# documents = output from PyMuPDFLoader
chunks = efficient_chunking(documents)


In [156]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata.setdefault("source", "residence_certificate")


In [157]:

client_embed = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1"
)


In [158]:
def embed_chunks(chunks):
    texts = [chunk.page_content for chunk in chunks]

    response = client_embed.embeddings.create(
        model="nvidia/nv-embedqa-e5-v5",
        input=texts,
        extra_body={"input_type": "passage"}
    )

    return [item.embedding for item in response.data]


In [159]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)


In [160]:
COLLECTION_NAME = "residence_certificate"

qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE
    )
)


/tmp/ipykernel_7793/2165392733.py:3: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(


True

In [161]:
from qdrant_client.models import PointStruct
import uuid

embeddings = embed_chunks(chunks)

points = []

for chunk, vector in zip(chunks, embeddings):
    points.append(
        PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload={
                "text": chunk.page_content,
                "metadata": chunk.metadata
            }
        )
    )

qdrant_client.upsert(
    collection_name=COLLECTION_NAME,
    points=points
)


UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [162]:
def retrieve_chunks(query: str, top_k: int = 6):
    # 1. Embed the query (QUERY MODE is IMPORTANT)
    query_embedding = client_embed.embeddings.create(
        model="nvidia/nv-embedqa-e5-v5",
        input=query,
        extra_body={"input_type": "query"}
    ).data[0].embedding

    # 2. Query Qdrant (NEW API)
    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=top_k
    )

    # 3. Extract stored text
    return [point.payload["text"] for point in results.points]

In [163]:
chunks = retrieve_chunks(
    "What documents are mandatory for residence certificate?"
)

for c in chunks:
    print("-" * 40)
    print(c[:300])


----------------------------------------
Mandatory Documents (Required for All Applicants)
The following documents must be submitted by every applicant without exception:
1. Applicant Photograph
2. Current Address Proof
3. Self-Declaration of Applicant
----------------------------------------
What is the Residence Certificate Service?
The Residence Certificate is an official government document issued by the Revenue
Administration department to certify that an individual resides at a specific address within a defined
jurisdiction. It is commonly used for accessing government schemes, edu
----------------------------------------
Important Notes
• Mandatory documents are required for all applicants.
• Supporting documents vary based on applicant category.
• Applicants are not required to submit all listed documents, only those applicable.
----------------------------------------
Residence Certificate – Structured Information
----------------------------------------
Service Details
Servic

In [164]:
client_llm = OpenAI(
    api_key=NVIDIA_LLM_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1"
)
print(client_llm.models.list())


SyncPage[Model](data=[Model(id='01-ai/yi-large', created=735790403, object='model', owned_by='01-ai'), Model(id='abacusai/dracarys-llama-3.1-70b-instruct', created=735790403, object='model', owned_by='abacusai'), Model(id='adept/fuyu-8b', created=735790403, object='model', owned_by='adept'), Model(id='ai21labs/jamba-1.5-large-instruct', created=735790403, object='model', owned_by='ai21labs'), Model(id='ai21labs/jamba-1.5-mini-instruct', created=735790403, object='model', owned_by='ai21labs'), Model(id='aisingapore/sea-lion-7b-instruct', created=735790403, object='model', owned_by='aisingapore'), Model(id='baai/bge-m3', created=735790403, object='model', owned_by='baai'), Model(id='baichuan-inc/baichuan2-13b-chat', created=735790403, object='model', owned_by='baichuan-inc'), Model(id='bigcode/starcoder2-15b', created=735790403, object='model', owned_by='bigcode'), Model(id='bigcode/starcoder2-7b', created=735790403, object='model', owned_by='bigcode'), Model(id='bytedance/seed-oss-36b-i

In [165]:
def rag_answer(question: str, context: str):
    response = client_llm.chat.completions.create(
        model="meta/llama-3.1-8b-instruct",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a government service assistant. "
                    "You MUST answer strictly using the provided context. "
                    "DO NOT infer, estimate, or modify any numbers. "
                    "If a numeric value appears in the context, reproduce it exactly. "
                    "If the context contains a list of documents, summarise the list clearly. "
                    "If the answer cannot be answered using the context, say exactly: "
                    "'The information is not available in the provided documents.'"
                )
            },
            {
                "role": "user",
                "content": f"""
Use ONLY the information in the Context section.

Context:
{context}

Question:
{question}

Answer format:
- Give a single, direct answer.
- Do not add extra explanation.
"""
            }
        ],
        temperature=0.2,
        max_tokens=512
    )

    return response.choices[0].message.content


In [166]:
def answer_question(question: str):
    contexts = retrieve_chunks(question, top_k=6)
    context_text = "\n\n".join(contexts)
    return rag_answer(question, context_text)


In [174]:
print(answer_question("Why i have to submit these documents for getting my residence certificate?"))


Documents are collected to verify the applicant's identity, confirm their residential address, and prevent misuse or fraudulent claims.
